<a href="https://colab.research.google.com/github/leatkePT/YoloModels/blob/master/TimeEstimations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q keras-cv pycocotools


from google.colab import drive
drive.mount('/content/drive')


print("unzipping , might take a while")
!mkdir -p /content/datasets/global_wheat
!mkdir -p /content/datasets/pascal_voc
!mkdir -p /content/datasets/oxford_pets

!unzip -q -o "/content/drive/MyDrive/YOLO_Project/global_wheat.zip" -d "/content/datasets/global_wheat"
!unzip -q -o "/content/drive/MyDrive/YOLO_Project/pascal_voc.zip" -d "/content/datasets/pascal_voc"
!unzip -q -o "/content/drive/MyDrive/YOLO_Project/oxford_pets.zip" -d "/content/datasets/oxford_pets"


!cp "/content/drive/MyDrive/YOLO_Project/yolo_model.py" "/content/"

print(" everything is ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 48.5 MB/s eta 0:00:00
Mounted at /content/drive
Ξεκινάει η αποσυμπίεση. Παρακαλώ περίμενε 1-2 λεπτά...
✅ Όλα τα δεδομένα είναι έτοιμα και ξεζιπαρισμένα στον τοπικό δίσκο του Colab!


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import xml.etree.ElementTree as ET
import keras_cv
import csv
import ast
from tqdm import tqdm

from yolo_model import YOLOv7_Tiny_OfficialMatch


# 1. DATASET SWITCHER & SETTINGS

TARGET_SIZE = (416, 416)
BATCH_SIZE = 16
EPOCHS_PER_EXPERIMENT = 1
LEARNING_RATE = 1e-4

# remove the comments on the  DATASET you want to train

#   OXFORD PETS
# DATASET_NAME = "Oxford Pets"
# DATASET_TYPE = "XML"
# IMAGE_DIR = "/content/datasets/oxford_pets/images"
# ANNOT_PATH = "/content/datasets/oxford_pets/annotations/xmls"
# CLASSES = ["dog", "cat"]

#    PASCAL VOC
# DATASET_NAME = "PASCAL VOC"
# DATASET_TYPE = "XML"
# IMAGE_DIR = "/content/datasets/pascal_voc/VOCdevkit/VOC2007/JPEGImages"
# ANNOT_PATH = "/content/datasets/pascal_voc/VOCdevkit/VOC2007/Annotations"
# CLASSES = [
#      "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
#      "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
#      "pottedplant", "sheep", "sofa", "train", "tvmonitor"
#  ]

#   GLOBAL WHEAT HEADS
DATASET_NAME = "Global Wheat"
DATASET_TYPE = "CSV"
IMAGE_DIR = "/content/datasets/global_wheat/train"
ANNOT_PATH = "/content/drive/MyDrive/YOLO_Project/train.csv"
CLASSES = ["wheat_head"]


NUM_CLASSES = len(CLASSES)
class_to_id = {name.lower(): i for i, name in enumerate(CLASSES)}

# 2. DATASET GENERATOR
def dataset_generator():
    if DATASET_TYPE == "CSV":
        grouped_boxes = {}
        with open(ANNOT_PATH, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                img_id = row['image_id']
                if img_id not in grouped_boxes:
                    grouped_boxes[img_id] = []
                bbox = ast.literal_eval(row['bbox'])
                orig_w, orig_h = float(row['width']), float(row['height'])
                xmin, ymin, w, h = bbox
                xc, yc = (xmin + w / 2.0) / orig_w, (ymin + h / 2.0) / orig_h
                w_norm, h_norm = w / orig_w, h / orig_h
                grouped_boxes[img_id].append([xc, yc, w_norm, h_norm])

        image_ids = list(grouped_boxes.keys())
        split_index = int(len(image_ids) * 0.8)

        for i, img_id in enumerate(image_ids):
            img_path = os.path.join(IMAGE_DIR, img_id + ".jpg")
            if not os.path.exists(img_path): continue

            image = tf.io.read_file(img_path)
            image = tf.image.decode_jpeg(image, channels=3)
            image = tf.cast(image, tf.float32) / 255.0
            image = tf.image.resize(image, TARGET_SIZE)

            boxes = grouped_boxes[img_id]
            current_classes = [0] * len(boxes)
            boxes_padded = np.zeros((50, 4), dtype=np.float32)
            classes_padded = np.zeros((50,), dtype=np.float32) - 1
            num_boxes = min(len(boxes), 50)
            boxes_padded[:num_boxes] = np.array(boxes[:num_boxes], dtype=np.float32)
            classes_padded[:num_boxes] = np.array(current_classes[:num_boxes], dtype=np.float32)
            yield image, boxes_padded, classes_padded, (0 if i < split_index else 1)

    elif DATASET_TYPE == "XML":
        image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith('.jpg')]
        split_index = int(len(image_files) * 0.8)

        for i, img_file in enumerate(image_files):
            img_path = os.path.join(IMAGE_DIR, img_file)
            xml_name = img_file.rsplit('.', 1)[0] + '.xml'
            xml_path = os.path.join(ANNOT_PATH, xml_name)

            if not os.path.exists(xml_path): continue

            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                orig_w = float(root.find('size/width').text)
                orig_h = float(root.find('size/height').text)

                boxes, current_classes = [], []
                for obj in root.findall('object'):
                    name = obj.find('name').text.lower().strip()
                    if name not in class_to_id: continue
                    current_classes.append(class_to_id[name])
                    bndbox = obj.find('bndbox')
                    xmin = float(bndbox.find('xmin').text) / orig_w
                    ymin = float(bndbox.find('ymin').text) / orig_h
                    xmax = float(bndbox.find('xmax').text) / orig_w
                    ymax = float(bndbox.find('ymax').text) / orig_h
                    boxes.append([(xmin + xmax) / 2.0, (ymin + ymax) / 2.0, xmax - xmin, ymax - ymin])

                if len(boxes) == 0: continue

                image = tf.io.read_file(img_path)
                image = tf.image.decode_jpeg(image, channels=3)
                image = tf.cast(image, tf.float32) / 255.0
                image = tf.image.resize(image, TARGET_SIZE)

                boxes_padded = np.zeros((50, 4), dtype=np.float32)
                classes_padded = np.zeros((50,), dtype=np.float32) - 1
                num_boxes = min(len(boxes), 50)
                boxes_padded[:num_boxes] = np.array(boxes[:num_boxes], dtype=np.float32)
                classes_padded[:num_boxes] = np.array(current_classes[:num_boxes], dtype=np.float32)
                yield image, boxes_padded, classes_padded, (0 if i < split_index else 1)
            except Exception:
                continue

dataset = tf.data.Dataset.from_generator(
    dataset_generator,
    output_signature=(
        tf.TensorSpec(shape=(416, 416, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(50, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(50,), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).cache(f"/content/experiment_cache_{DATASET_NAME.replace(' ', '_')}")

train_dataset = dataset.filter(lambda img, box, cls, is_val: is_val == 0).map(
    lambda img, box, cls, is_val: (img, box, cls)).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

val_dataset = dataset.filter(lambda img, box, cls, is_val: is_val == 1).map(
    lambda img, box, cls, is_val: (img, box, cls)).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


# 3. YOLOV7 LOSS & DECODER

@tf.function
def compute_yolo_loss(true_boxes, true_classes, model_outputs):
    out_small, out_medium, out_large = model_outputs
    loss = 0.0
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    mse = tf.keras.losses.MeanSquaredError()
    for grid_pred in [out_small, out_medium, out_large]:
        pred_xy = tf.math.sigmoid(grid_pred[..., 0:2])
        pred_obj = grid_pred[..., 4:5]
        loss += bce(tf.zeros_like(pred_obj), pred_obj)
        loss += mse(true_boxes[:, 0, 0:2], tf.reduce_mean(pred_xy, axis=[1, 2])) * 5.0
    return loss / 3.0

@tf.function
def decode_yolo_outputs(model_outputs):
    all_boxes, all_classes, all_conf = [], [], []
    for grid in model_outputs:
        batch_size = tf.shape(grid)[0]
        grid_flat = tf.reshape(grid, [batch_size, -1, NUM_CLASSES + 5])
        xy = tf.math.sigmoid(grid_flat[..., 0:2])
        wh = tf.math.exp(grid_flat[..., 2:4])
        boxes = tf.concat([xy, wh], axis=-1)
        obj_conf = tf.math.sigmoid(grid_flat[..., 4:5])
        class_probs = tf.math.sigmoid(grid_flat[..., 5:])
        best_class = tf.argmax(class_probs, axis=-1)
        best_class_score = tf.reduce_max(class_probs, axis=-1)
        final_conf = tf.squeeze(obj_conf, axis=-1) * best_class_score
        all_boxes.append(boxes)
        all_classes.append(tf.cast(best_class, tf.float32))
        all_conf.append(final_conf)

    final_boxes = tf.concat(all_boxes, axis=1)
    final_classes = tf.concat(all_classes, axis=1)
    final_conf = tf.concat(all_conf, axis=1)

    top_k = tf.minimum(100, tf.shape(final_conf)[1])
    top_conf, top_indices = tf.math.top_k(final_conf, k=top_k)
    final_boxes_top = tf.gather(final_boxes, top_indices, batch_dims=1)
    final_classes_top = tf.gather(final_classes, top_indices, batch_dims=1)

    return {"boxes": final_boxes_top, "classes": final_classes_top, "confidence": top_conf}


# 4.EXPERIMENT RUNNER

def run_experiment(experiment_name, use_map, num_epochs, log_file):
    print(f"\n" + "="*50)
    print(f" Start experiment: {experiment_name}")
    print("="*50)


    tf.keras.backend.clear_session()
    model = YOLOv7_Tiny_OfficialMatch(input_shape=(416, 416, 3), num_classes=NUM_CLASSES)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

    val_map_metric = None
    if use_map:
        val_map_metric = keras_cv.metrics.BoxCOCOMetrics(bounding_box_format="xywh", evaluate_freq=1)

    @tf.function
    def train_step(images, boxes, classes):
        with tf.GradientTape() as tape:
            predictions = model(images, training=True)
            loss = compute_yolo_loss(boxes, classes, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss

    for epoch in range(num_epochs):
        print(f"\n--- {experiment_name} | Epoch {epoch + 1}/{num_epochs} ---")

        #  TRAINING PHASE
        train_start_time = time.time()
        train_loss_total = 0.0
        num_train_batches = 0
        pbar_train = tqdm(train_dataset, desc="Training", leave=False)

        for batch_images, batch_boxes, batch_classes in pbar_train:
            loss = train_step(batch_images, batch_boxes, batch_classes)
            train_loss_total += float(loss)
            num_train_batches += 1
            pbar_train.set_postfix(Loss=f"{train_loss_total/num_train_batches:.4f}")

        train_duration = time.time() - train_start_time
        avg_train_loss = train_loss_total / max(1, num_train_batches)

        #  VALIDATION PHASE
        val_start_time = time.time()
        val_loss_total = 0.0
        num_val_batches = 0
        if use_map: val_map_metric.reset_state()

        pbar_val = tqdm(val_dataset, desc="Validation", leave=False)

        for val_images, val_boxes, val_classes in pbar_val:
            val_predictions = model(val_images, training=False)
            val_loss = compute_yolo_loss(val_boxes, val_classes, val_predictions)
            val_loss_total += float(val_loss)
            num_val_batches += 1

            if use_map:
                y_pred_dict = decode_yolo_outputs(val_predictions)
                val_map_metric.update_state({"boxes": val_boxes, "classes": val_classes}, y_pred_dict)

            pbar_val.set_postfix(Val_Loss=f"{val_loss_total/num_val_batches:.4f}")

        val_duration = time.time() - val_start_time
        avg_val_loss = val_loss_total / max(1, num_val_batches)
        total_duration = train_duration + val_duration

        #  EXTRACT METRICS
        map_50 = 0.0
        map_50_95 = 0.0

        if use_map:
            current_map = val_map_metric.result()
            if experiment_name == "With_mAP_0.5":
                map_50 = float(current_map['MaP@[IoU=50]'])
            elif experiment_name == "With_mAP_0.5_0.95":
                map_50 = float(current_map['MaP@[IoU=50]'])
                map_50_95 = float(current_map['MaP'])

        # Console Output
        print(f" Total Time: {total_duration:.2f}s (Train: {train_duration:.2f}s | Val: {val_duration:.2f}s)")
        print(f" Loss -> Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
        if use_map:
            print(f" mAP@0.5: {map_50:.4f} | mAP@0.5:0.95: {map_50_95:.4f}")

        # Save to CSV
        with open(log_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                experiment_name, epoch + 1,
                round(train_duration, 2), round(val_duration, 2), round(total_duration, 2),
                round(avg_train_loss, 4), round(avg_val_loss, 4),
                round(map_50, 4), round(map_50_95, 4)
            ])


# 5.  EXECUTION SCRIPT

if __name__ == "__main__":
    LOG_FILE = f"/content/drive/MyDrive/YOLO_Project/epoch_experiments_{DATASET_NAME.replace(' ', '_')}.csv"


    if not os.path.exists(LOG_FILE):
        with open(LOG_FILE, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Experiment", "Epoch", "Train_Time_s", "Val_Time_s", "Total_Time_s", "Train_Loss", "Val_Loss", "mAP_50", "mAP_50_95"])

    print("=" * 60)
    print(f"=== ΞΕΚΙΝΑΕΙ Η ΑΥΤΟΜΑΤΗ ΔΟΚΙΜΗ 3 ΣΤΑΔΙΩΝ ΓΙΑ: {DATASET_NAME} ===")
    print("=" * 60)

    # RUN 1:  Without mAP
    run_experiment(experiment_name="Without_mAP", use_map=False, num_epochs=EPOCHS_PER_EXPERIMENT, log_file=LOG_FILE)

    # RUN 2:  With mAP@0.5
    run_experiment(experiment_name="With_mAP_0.5", use_map=True, num_epochs=EPOCHS_PER_EXPERIMENT, log_file=LOG_FILE)

    # RUN 3:  With BOTH mAPs
    run_experiment(experiment_name="With_mAP_0.5_0.95", use_map=True, num_epochs=EPOCHS_PER_EXPERIMENT, log_file=LOG_FILE)

    print(f"\n ALL data saved at: {LOG_FILE}")

=== ΞΕΚΙΝΑΕΙ Η ΑΥΤΟΜΑΤΗ ΔΟΚΙΜΗ 3 ΣΤΑΔΙΩΝ ΓΙΑ: Global Wheat ===

🚀 ΕΚΚΙΝΗΣΗ ΠΕΙΡΑΜΑΤΟΣ: With_mAP_0.5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(



--- With_mAP_0.5 | Epoch 1/10 ---


✅ Total Time: 1323.59s (Train: 141.73s | Val: 1181.86s)
📊 Loss -> Train: 0.6841 | Val: 0.9043
🎯 mAP@0.5: 0.0000 | mAP@0.5:0.95: 0.0000

--- With_mAP_0.5 | Epoch 2/10 ---


✅ Total Time: 1204.47s (Train: 67.15s | Val: 1137.32s)
📊 Loss -> Train: 0.5130 | Val: 0.8825
🎯 mAP@0.5: 0.0000 | mAP@0.5:0.95: 0.0000

--- With_mAP_0.5 | Epoch 3/10 ---


✅ Total Time: 1231.57s (Train: 68.98s | Val: 1162.59s)
📊 Loss -> Train: 0.4747 | Val: 0.7370
🎯 mAP@0.5: 0.0000 | mAP@0.5:0.95: 0.0000

--- With_mAP_0.5 | Epoch 4/10 ---


✅ Total Time: 1236.67s (Train: 68.93s | Val: 1167.74s)
📊 Loss -> Train: 0.4094 | Val: 0.7837
🎯 mAP@0.5: 0.0000 | mAP@0.5:0.95: 0.0000

--- With_mAP_0.5 | Epoch 5/10 ---


✅ Total Time: 1281.55s (Train: 68.61s | Val: 1212.94s)
📊 Loss -> Train: 0.3218 | Val: 0.9974
🎯 mAP@0.5: 0.0000 | mAP@0.5:0.95: 0.0000

--- With_mAP_0.5 | Epoch 6/10 ---


Validation: 31it [11:15, 37.66s/it, Val_Loss=1.0905]